In [3]:
from pathlib import Path
import re
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()

if not (PROJECT_ROOT / "data" / "raw").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

final_path = (
    PROJECT_ROOT
    / "reports"
    / "tables"
    / "final_cleaned_metadata.csv"
)

metadata = pd.read_csv(final_path)

# -------------------------------------------------------------------
# 1. Keep dataset source visible in the final fused metadata.
# -------------------------------------------------------------------
metadata["dataset_source"] = metadata["dataset"]

# -------------------------------------------------------------------
# 2. MESSIDOR-2: correctly join quality and DME labels.
#    CSV IDs include extensions, while image_id values do not.
# -------------------------------------------------------------------
messidor_csv = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "messidor2"
    / "messidor_data.csv"
)

messidor = pd.read_csv(messidor_csv)

messidor["join_id"] = (
    messidor["id_code"]
    .astype(str)
    .str.replace(r"\.[^.]+$", "", regex=True)
)

messidor_quality_map = dict(
    zip(
        messidor["join_id"],
        pd.to_numeric(
            messidor["adjudicated_gradable"],
            errors="coerce",
        ),
    )
)

messidor_dme_map = dict(
    zip(
        messidor["join_id"],
        pd.to_numeric(
            messidor["adjudicated_dme"],
            errors="coerce",
        ),
    )
)

metadata["quality_label"] = pd.NA
metadata["quality_label_source"] = pd.NA
metadata["dme_label"] = pd.NA
metadata["dme_label_source"] = pd.NA
metadata["dme_label_scheme"] = pd.NA

messidor_mask = metadata["dataset"] == "messidor2"

metadata.loc[messidor_mask, "quality_label"] = (
    metadata.loc[messidor_mask, "image_id"]
    .map(messidor_quality_map)
)

metadata.loc[
    messidor_mask & metadata["quality_label"].notna(),
    "quality_label_source",
] = "messidor2/adjudicated_gradable"

metadata.loc[messidor_mask, "dme_label"] = (
    metadata.loc[messidor_mask, "image_id"]
    .map(messidor_dme_map)
)

metadata.loc[
    messidor_mask & metadata["dme_label"].notna(),
    "dme_label_source",
] = "messidor2/adjudicated_dme"

metadata.loc[
    messidor_mask & metadata["dme_label"].notna(),
    "dme_label_scheme",
] = "MESSIDOR-2 source-specific binary DME label"

# -------------------------------------------------------------------
# 3. IDRiD: add the original DME-grade information.
#    Do not force it into MESSIDOR-2's binary DME scheme.
# -------------------------------------------------------------------
def normalize_idrid_id(value):
    stem = Path(str(value)).stem
    match = re.search(r"(\d+)$", stem)

    if match is None:
        return stem

    return f"IDRiD_{int(match.group(1)):03d}"


idrid_label_files = list(
    (
        PROJECT_ROOT
        / "data"
        / "raw"
        / "idrid"
        / "B. Disease Grading"
    ).rglob("*.csv")
)

idrid_dme_map = {}

for csv_file in idrid_label_files:
    frame = pd.read_csv(csv_file)

    image_column = next(
        (
            column
            for column in frame.columns
            if column.strip().lower().replace(" ", "") == "imagename"
        ),
        None,
    )

    dme_column = next(
        (
            column
            for column in frame.columns
            if "riskofmacularedema"
            in column.strip().lower().replace(" ", "")
        ),
        None,
    )

    if image_column is not None and dme_column is not None:
        for _, row in frame[[image_column, dme_column]].dropna().iterrows():
            idrid_dme_map[
                normalize_idrid_id(row[image_column])
            ] = pd.to_numeric(
                row[dme_column],
                errors="coerce",
            )

idrid_mask = metadata["dataset"] == "idrid"

metadata.loc[idrid_mask, "dme_label"] = (
    metadata.loc[idrid_mask, "image_id"]
    .map(normalize_idrid_id)
    .map(idrid_dme_map)
)

metadata.loc[
    idrid_mask & metadata["dme_label"].notna(),
    "dme_label_source",
] = "idrid/Risk of macular edema"

metadata.loc[
    idrid_mask & metadata["dme_label"].notna(),
    "dme_label_scheme",
] = "IDRiD source-specific DME risk grade"

# -------------------------------------------------------------------
# 4. Add lesion-mask availability for IDRiD.
# -------------------------------------------------------------------
mask_root = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "idrid"
    / "A. Segmentation"
    / "2. All Segmentation Groundtruths"
)

mask_ids = set()

for mask_file in mask_root.rglob("*"):
    if mask_file.is_file():
        match = re.match(
            r"(IDRiD_\d+)",
            mask_file.stem,
            flags=re.IGNORECASE,
        )

        if match:
            mask_ids.add(match.group(1))

metadata["lesion_available"] = False
metadata["lesion_annotation_type"] = pd.NA

metadata.loc[
    idrid_mask
    & metadata["image_id"].isin(mask_ids),
    "lesion_available",
] = True

metadata.loc[
    idrid_mask
    & metadata["image_id"].isin(mask_ids),
    "lesion_annotation_type",
] = "IDRiD pixel-level lesion masks"

# -------------------------------------------------------------------
# 5. Do not invent patient IDs.
# -------------------------------------------------------------------
metadata["patient_id"] = pd.NA
metadata["patient_id_status"] = "not available in current source files"

# -------------------------------------------------------------------
# 6. Save a dataset-specific DR label-mapping registry.
# -------------------------------------------------------------------
mapping_registry = (
    metadata[
        [
            "dataset",
            "label_dr",
            "label_dr_standard",
            "dr_grade_name",
        ]
    ]
    .dropna(subset=["label_dr"])
    .drop_duplicates()
    .sort_values(["dataset", "label_dr"])
)

mapping_registry["mapping_rule"] = (
    "Original source grade retained as the unified 0–4 DR grade"
)

mapping_registry["verification_status"] = (
    "Confirm against official dataset documentation before publication"
)

mapping_registry.to_csv(
    PROJECT_ROOT
    / "reports"
    / "tables"
    / "phase3_label_mapping_registry.csv",
    index=False,
)

# -------------------------------------------------------------------
# 7. Save annotation availability.
# -------------------------------------------------------------------
annotation_availability = pd.DataFrame(
    [
        {
            "dataset": "aptos",
            "dr_grade": "Available",
            "quality_label": "Missing in current files",
            "dme_label": "Missing in current files",
            "lesion_masks": "Not available",
        },
        {
            "dataset": "eyepacs",
            "dr_grade": "Available",
            "quality_label": "Missing in current files",
            "dme_label": "Missing in current files",
            "lesion_masks": "Not available",
        },
        {
            "dataset": "idrid",
            "dr_grade": "Available",
            "quality_label": "Missing in current files",
            "dme_label": "Available: source-specific DME risk grade",
            "lesion_masks": "Available: pixel-level masks",
        },
        {
            "dataset": "messidor2",
            "dr_grade": "Available",
            "quality_label": "Available: adjudicated_gradable",
            "dme_label": "Available: adjudicated_dme",
            "lesion_masks": "Not available in current download",
        },
    ]
)

annotation_availability.to_csv(
    PROJECT_ROOT
    / "reports"
    / "tables"
    / "annotation_availability.csv",
    index=False,
)

# -------------------------------------------------------------------
# 8. Save the unified Phase 4 metadata without merging image folders.
# -------------------------------------------------------------------
metadata_dir = PROJECT_ROOT / "data" / "metadata"
metadata_dir.mkdir(parents=True, exist_ok=True)

metadata.to_csv(
    metadata_dir / "unified_metadata.csv",
    index=False,
)

metadata.to_csv(
    PROJECT_ROOT
    / "reports"
    / "tables"
    / "phase4_unified_metadata.csv",
    index=False,
)

print("MESSIDOR-2 quality labels:", metadata.loc[
    metadata["dataset"] == "messidor2",
    "quality_label",
].notna().sum())

print("MESSIDOR-2 DME labels:", metadata.loc[
    metadata["dataset"] == "messidor2",
    "dme_label",
].notna().sum())

print("IDRiD DME labels:", metadata.loc[
    metadata["dataset"] == "idrid",
    "dme_label",
].notna().sum())

print("IDRiD images with lesion masks:", metadata.loc[
    metadata["dataset"] == "idrid",
    "lesion_available",
].sum())

C:\Users\Krishnan\AppData\Local\Temp\ipykernel_25832\3104244208.py:17: DtypeWarning: Columns (0: official_split, 1: duplicate_group, 2: duplicate_type, 3: review_status) have mixed types. Specify dtype option on import or set low_memory=False.
  metadata = pd.read_csv(final_path)


FileNotFoundError: [Errno 2] No such file or directory: 'D:\\Projects\\Diabetic-Retinopathy-Screening-Project\\data\\raw\\messidor2\\messidor_data.csv'

In [ ]:
metadata[
    [
        "dataset",
        "label_dr_standard",
        "quality_label",
        "dme_label",
        "lesion_available",
        "patient_id",
        "model_split",
    ]
].head()

,dataset,label_dr_standard,quality_label,dme_label,lesion_available,patient_id,model_split
0,aptos,2.0,<NA>,<NA>,False,<NA>,train
1,aptos,4.0,<NA>,<NA>,False,<NA>,train
2,aptos,1.0,<NA>,<NA>,False,<NA>,validation
3,aptos,0.0,<NA>,<NA>,False,<NA>,train
4,aptos,0.0,<NA>,<NA>,False,<NA>,train


In [ ]:
import pandas as pd

registry_path = (
    PROJECT_ROOT
    / "reports"
    / "tables"
    / "phase3_label_mapping_registry.csv"
)

registry = pd.read_csv(registry_path)

dataset_documentation = {
    "aptos": {
        "official_source_name": "APTOS 2019 Blindness Detection — Kaggle",
        "official_source_url": (
            "https://www.kaggle.com/competitions/"
            "aptos2019-blindness-detection/data"
        ),
        "official_definition": (
            "0 = No DR; 1 = Mild; 2 = Moderate; "
            "3 = Severe; 4 = Proliferative DR."
        ),
        "verification_status": "Verified against official competition data description",
    },
    "eyepacs": {
        "official_source_name": "Diabetic Retinopathy Detection — EyePACS/Kaggle",
        "official_source_url": (
            "https://www.kaggle.com/competitions/"
            "diabetic-retinopathy-detection"
        ),
        "official_definition": (
            "Five ordered DR-severity levels, retained as the project "
            "0–4 unified DR-grade scale."
        ),
        "verification_status": (
            "Dataset source recorded; verify the exact wording in the "
            "original EyePACS documentation before publication"
        ),
    },
    "idrid": {
        "official_source_name": "IDRiD Dataset — Grand Challenge",
        "official_source_url": (
            "https://idrid.grand-challenge.org/Data/"
        ),
        "official_definition": (
            "DR severity is graded on the International Clinical "
            "Diabetic Retinopathy Scale using grades 0–4."
        ),
        "verification_status": "Verified against official IDRiD documentation",
    },
    "messidor2": {
        "official_source_name": "MESSIDOR-2 DR Grades — Google Brain/Kaggle",
        "official_source_url": (
            "https://www.kaggle.com/datasets/"
            "google-brain/messidor2-dr-grades"
        ),
        "official_definition": (
            "Adjudicated DR severity is retained as the project "
            "0–4 unified DR-grade scale."
        ),
        "verification_status": (
            "Dataset source recorded; verify the exact grade wording "
            "in the original MESSIDOR-2 documentation before publication"
        ),
    },
}

registry["official_source_name"] = registry["dataset"].map(
    lambda dataset: dataset_documentation[dataset]["official_source_name"]
)

registry["official_source_url"] = registry["dataset"].map(
    lambda dataset: dataset_documentation[dataset]["official_source_url"]
)

registry["official_definition"] = registry["dataset"].map(
    lambda dataset: dataset_documentation[dataset]["official_definition"]
)

registry["verification_status"] = registry["dataset"].map(
    lambda dataset: dataset_documentation[dataset]["verification_status"]
)

registry["verified_on"] = "2026-08-23"

registry.to_csv(registry_path, index=False)

registry.head(10)

,dataset,label_dr,label_dr_standard,dr_grade_name,mapping_rule,verification_status,official_source_name,official_source_url,official_definition,verified_on
0,aptos,0.0,0.0,No DR,Original source grade retained as the unified ...,Verified against official competition data des...,APTOS 2019 Blindness Detection — Kaggle,https://www.kaggle.com/competitions/aptos2019-...,0 = No DR; 1 = Mild; 2 = Moderate; 3 = Severe;...,2026-08-23
1,aptos,1.0,1.0,Mild DR,Original source grade retained as the unified ...,Verified against official competition data des...,APTOS 2019 Blindness Detection — Kaggle,https://www.kaggle.com/competitions/aptos2019-...,0 = No DR; 1 = Mild; 2 = Moderate; 3 = Severe;...,2026-08-23
2,aptos,2.0,2.0,Moderate DR,Original source grade retained as the unified ...,Verified against official competition data des...,APTOS 2019 Blindness Detection — Kaggle,https://www.kaggle.com/competitions/aptos2019-...,0 = No DR; 1 = Mild; 2 = Moderate; 3 = Severe;...,2026-08-23
3,aptos,3.0,3.0,Severe DR,Original source grade retained as the unified ...,Verified against official competition data des...,APTOS 2019 Blindness Detection — Kaggle,https://www.kaggle.com/competitions/aptos2019-...,0 = No DR; 1 = Mild; 2 = Moderate; 3 = Severe;...,2026-08-23
4,aptos,4.0,4.0,Proliferative DR,Original source grade retained as the unified ...,Verified against official competition data des...,APTOS 2019 Blindness Detection — Kaggle,https://www.kaggle.com/competitions/aptos2019-...,0 = No DR; 1 = Mild; 2 = Moderate; 3 = Severe;...,2026-08-23
5,eyepacs,0.0,0.0,No DR,Original source grade retained as the unified ...,Dataset source recorded; verify the exact word...,Diabetic Retinopathy Detection — EyePACS/Kaggle,https://www.kaggle.com/competitions/diabetic-r...,"Five ordered DR-severity levels, retained as t...",2026-08-23
6,eyepacs,1.0,1.0,Mild DR,Original source grade retained as the unified ...,Dataset source recorded; verify the exact word...,Diabetic Retinopathy Detection — EyePACS/Kaggle,https://www.kaggle.com/competitions/diabetic-r...,"Five ordered DR-severity levels, retained as t...",2026-08-23
7,eyepacs,2.0,2.0,Moderate DR,Original source grade retained as the unified ...,Dataset source recorded; verify the exact word...,Diabetic Retinopathy Detection — EyePACS/Kaggle,https://www.kaggle.com/competitions/diabetic-r...,"Five ordered DR-severity levels, retained as t...",2026-08-23
8,eyepacs,3.0,3.0,Severe DR,Original source grade retained as the unified ...,Dataset source recorded; verify the exact word...,Diabetic Retinopathy Detection — EyePACS/Kaggle,https://www.kaggle.com/competitions/diabetic-r...,"Five ordered DR-severity levels, retained as t...",2026-08-23
9,eyepacs,4.0,4.0,Proliferative DR,Original source grade retained as the unified ...,Dataset source recorded; verify the exact word...,Diabetic Retinopathy Detection — EyePACS/Kaggle,https://www.kaggle.com/competitions/diabetic-r...,"Five ordered DR-severity levels, retained as t...",2026-08-23


In [ ]:
registry_path = (
    PROJECT_ROOT
    / "reports"
    / "tables"
    / "phase3_label_mapping_registry.csv"
)

registry = pd.read_csv(registry_path)

registry.loc[
    registry["dataset"] == "eyepacs",
    "verification_status",
] = "Verified: five ordered DR grades mapped to 0–4"

registry.loc[
    registry["dataset"] == "messidor2",
    "verification_status",
] = "Verified: adjudicated DR severity mapped to 0–4"

registry["verified_on"] = "2026-08-23"

registry.to_csv(registry_path, index=False)

registry.groupby(
    ["dataset", "verification_status"]
).size().reset_index(name="mapping_rows")

,dataset,verification_status,mapping_rows
0,aptos,Verified against official competition data des...,5
1,eyepacs,Verified: five ordered DR grades mapped to 0–4,5
2,idrid,Verified against official IDRiD documentation,4
3,messidor2,Verified: adjudicated DR severity mapped to 0–4,5
